# Phishing Detector Starter Notebook
This notebook catalogs the Zenodo files, builds a label map, and trains a small TF‑IDF baseline.
> Record: Zenodo 8339691 — *Phishing Email Curated Datasets*

In [ ]:

import os, json, pandas as pd, numpy as np, re, textwrap, joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline

META_PATH = r"/mnt/data/8339691.json"
OUT_DIR = r"/mnt/data/phishing_starter"
os.makedirs(OUT_DIR, exist_ok=True)

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

def extract_files(m):
    files = []
    def get_files(d):
        if isinstance(d, dict):
            for k,v in d.items():
                if k.lower()=="files" and isinstance(v, list):
                    for it in v:
                        yield it
                elif isinstance(v, (dict, list)):
                    yield from get_files(v)
        elif isinstance(d, list):
            for x in d:
                yield from get_files(x)
    for it in get_files(m):
        name = it.get("key") or it.get("filename") or it.get("name")
        size = it.get("size") or it.get("filesize")
        checksum = it.get("checksum") or it.get("md5") or it.get("sha256")
        links = it.get("links") or {}
        download = links.get("self") or links.get("download") or it.get("download_url") or it.get("href")
        if name or size or checksum or download:
            files.append({"name": name, "size_bytes": size, "checksum": checksum, "download": download})
    import pandas as pd
    return pd.DataFrame(files)

files_df = extract_files(meta)
files_df["dataset_key"] = files_df["name"].astype(str).str.replace(r"\.csv(\.gz)?$", "", regex=True).str.replace("/", "_")
files_df


In [ ]:

def heuristic_label(dataset_key: str):
    dk = (dataset_key or "").lower()
    if any(tok in dk for tok in ["nazario","nigerian","419","phish"]):
        return {"label_raw":"phish","label_mapped":"phish","notes":"Assumed phishing corpus."}
    if any(tok in dk for tok in ["enron","ham"]):
        return {"label_raw":"ham","label_mapped":"not_phish","notes":"Assumed legitimate ham corpus."}
    if any(tok in dk for tok in ["spamassassin","spam_assassin","trec","ceas","spam"]):
        return {"label_raw":"spam","label_mapped":"not_phish","notes":"Assumed non-phishing spam."}
    return {"label_raw":"unknown","label_mapped":"unknown","notes":"Please verify."}

label_rows = []
for _, r in files_df.iterrows():
    info = heuristic_label(str(r.get("dataset_key","")))
    label_rows.append({
        "dataset_key": r.get("dataset_key"),
        "file_name": r.get("name"),
        "assumed_label_raw": info["label_raw"],
        "mapped_binary_label": info["label_mapped"],
        "notes": info["notes"]
    })
label_map_df = pd.DataFrame(label_rows)
label_map_path = os.path.join(OUT_DIR, "label_map.csv")
label_map_df.to_csv(label_map_path, index=False)
label_map_df


## Tiny TF‑IDF Baseline (Synthetic Sample)
Replace this with real parsed emails once you download the datasets.

In [ ]:

synthetic_samples = [
    ("Meeting today at 3pm. Please find the agenda attached.", 0),
    ("FYI the server will be restarted tonight. No action required.", 0),
    ("Lunch tomorrow? I can book a table near the office.", 0),
    ("Invoice for your purchase at our store last week.", 0),
    ("Weekly newsletter: product updates and blog highlights.", 0),
    ("Friendly reminder: submit your timesheet by EOD.", 0),
    ("URGENT: Verify your account now to avoid suspension. Click here to login.", 1),
    ("Your mailbox is almost full. Update your password immediately via the secure portal.", 1),
    ("We detected unusual sign-in activity. Confirm your identity at http://192.168.0.4-login.com", 1),
    ("Final notice: your package is on hold. Pay the customs fee to release it.", 1),
    ("Security alert: account locked. Reactivate by providing your credentials.", 1),
    ("Tax refund available. Submit bank info within 24 hours.", 1),
]
texts = [t for t,_ in synthetic_samples]
ys = [y for _,y in synthetic_samples]

X_train, X_test, y_train, y_test = train_test_split(texts, ys, test_size=0.33, random_state=42, stratify=ys)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3,5), max_features=20000)),
    ("lr", LogisticRegression(max_iter=200, class_weight="balanced"))
])
pipe.fit(X_train, y_train)
probs = pipe.predict_proba(X_test)[:,1]
preds = (probs>=0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, probs))
print("PR-AUC:", average_precision_score(y_test, probs))
print("\nClassification Report:\n", classification_report(y_test, preds, target_names=["not_phish","phish"]))

ART_DIR = os.path.join(OUT_DIR, "artifacts"); os.makedirs(ART_DIR, exist_ok=True)
joblib.dump(pipe, os.path.join(ART_DIR, "char_tfidf_logreg.joblib"))
